## Linear Regression Algorithm

Remember our linear regression algorithm:

repeat until convergence {
    w = w - a*djdw
    b = b - a*djdb
}

where:

djdw = sum of all ((wx[i] + b) - y[i])*x[i] divided by m, the number of training examples

djdb = sum of all (wx[i] + b) - y[i] divided by m, the number of training examples

Now that we understand the mathematics behind gradient descent, we can implement it in Python.

Let's start by writing a helper function called compute_gradient(). The purpose of this function is to calculate the two gradients:

djdw — the gradient of the cost function with respect to \(w\)
djdb — the gradient of the cost function with respect to b


Since we will be using numpy arrays to represent our data features and target values let's import the numpy library first.

In [15]:
import numpy as np

In [14]:
def compute_gradient(x,y,w,b):
    """
    Computes the gradient for linear regression 
    Args:
      x (ndarray (m,)): Data, m examples 
      y (ndarray (m,)): target values
      w,b (scalar)    : model parameters  
    Returns
      dj_dw (scalar): The gradient of the cost w.r.t. the parameters w
      dj_db (scalar): The gradient of the cost w.r.t. the parameter b     
     """
    m = x.shape[0]  # number of training examples
    djdw = 0
    djdb = 0

    for i in range(m):
        # Contribution of example i to the gradient with respect to w
        djdw_i = (w*x[i] + b - y[i])*x[i]
        djdw += djdw_i

        # Contribution of example i to the gradient with respect to b
        djdb_i = (w*x[i] + b - y[i])
        djdb += djdb_i
        
    # Calculate the average gradient
    djdw /= m
    djdb /= m
    return djdw, djdb

Now that we have a function that can calculate the gradients, we can use it to implement the gradient descent algorithm itself.

In [ ]:
def linear_regression(x,y,w_in,b_in,alpha,num_iters,compute_gradient):
    """
    Performs linear regression to learn w and b by taking num_iters gradient steps with learning rate alpha
    Args:
      x (ndarray (m,)): Data, m examples 
      y (ndarray (m,)): target values
      w_in,b_in (scalar) : initial values of model parameters
      alpha (float)      : learning rate
      num_iters (int)    : number of iterations to run gradient descent
      compute_gradient   : helper function that computes djdw and djdb
    Returns:
      w (scalar): Updated value of parameter after running gradient descent
      b (scalar): Updated value of parameter after running gradient descent
    """
    w = w_in
    b = b_in
    for _ in range(num_iters):
        djdw,djdb = compute_gradient(x,y,w,b)
        w = w - alpha*djdw
        b = b - alpha*djdb
    return w,b
        

One important observation is that compute_gradient() does not update \(w\) or \(b\). Its job is only to calculate the gradients. The linear_regression() function then uses those gradients to update the parameters. So we have separated the two responsibilities:

compute_gradient()
        ↓
calculates ∂J/∂w and ∂J/∂b
        ↓
linear_regression()
        ↓
updates w and b
        ↓
repeat

That separation is a good design choice because it makes each function responsible for one specific part of the algorithm.

## Detecting Convergence

One issue we have so far in our implementation is that we are making the assumption that after num_iters iterations, the model will have converged. If num_iters is too small, we may stop training before the model has converged. On the other hand, if num_iters is too large, we may continue performing unnecessary iterations after the model has already converged, wasting computation.

So instead of simply saying:

Repeat num_iters times

we would ideally like to say:

Repeat until convergence


## What does convergence mean?

In gradient descent, we can consider the algorithm to have converged when the cost function \(J(w,b)\) is no longer changing by a significant amount.

For example, suppose the cost after several iterations looks like this:

| Iteration |  Cost |
---------------------|
| 1         |   2500.00 |
| 2         |    1200.00|
| 3         |     650.00|
| 4         |     420.00|
| 5         |     350.00|
| 6         |     338.50|
| 7         |     337.90|
| 8         |     337.85|
| 9         |     337.84|
| 10        |     337.84|

As you can see, between iterations 8 and 9 the reduction in cost is only 0.01. If we decide that the reduction in cost is smaller than some defined thereshold, we can can stop training. This threshold, (e)  is called convergence tolerance.

Therefore, we can say:

If the difference between the costs at the previous iteration and the current iteration is below the convergence tolerance, stop the training 

That is if |J<sub>previous</sub> - J<sub>current</sub>| < ϵ then stop training.

where ϵ is a small positive number such as:

ϵ=10<sup>−6</sup>

Let's now implement this version of Linear Regression.